# ✈️ Aircraft Skin Defect Detection — Full Model Comparison

**Models evaluated:** YOLOv8 · YOLOv10 · YOLOv11 · YOLOv12 · RT-DETR · Faster R-CNN · Mask R-CNN

**Runtime:** GPU (T4 or better recommended)

---
### Notebook structure
1. Mount Google Drive & install dependencies
2. Upload / verify all model weights
3. Per-class metrics tables (YOLO-family + RT-DETR)
4. Faster R-CNN & Mask R-CNN evaluation (torchvision)
5. Aggregate comparison bar-chart
6. Macro-average AUC-ROC curves

## 1 · Mount Drive & Install Dependencies

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Install / upgrade required packages
# ultralytics covers YOLOv8/10/11/12 and RT-DETR
!pip install -q ultralytics==8.4.14 scikit-learn torchvision pycocotools

# Verify GPU
import torch
print(f'PyTorch {torch.__version__}  |  CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.0 MB/s eta 0:00:00
PyTorch 2.10.0+cu128  |  CUDA available: True
GPU: Tesla T4


## 2 · Configuration — Edit these paths to match your Drive layout

In [3]:
import os

# ── Root of your project folder inside Google Drive ───────────────────────────
BASE = '/content/drive/MyDrive/Project Exhibition'

# ── Full combined dataset YAML (4 557 images across all splits) ───────────────
FULL_DATA_YAML = os.path.join(BASE, '_full_dataset_temp', 'full_data.yaml')

# ── Validation label dir used for ROC (YOLO-format .txt files) ───────────────
VALID_LABEL_DIR = os.path.join(
    BASE,
    'YOLOv8/aircraft-skin-defects.v4-classification-isolated-5-classes-grayscale-prep.yolov8/valid/labels'
)

# ── COCO-format annotations for Faster/Mask R-CNN ────────────────────────────
COCO_VALID_JSON = os.path.join(
    BASE,
    'Masked RCNN/Masked RCNN/aircraft-skin-defects-new-dataset-4/valid/_annotations.coco.json'
)
COCO_VALID_IMAGES = os.path.join(
    BASE,
    'Masked RCNN/Masked RCNN/aircraft-skin-defects-new-dataset-4/valid'
)

# ── Model weight paths ────────────────────────────────────────────────────────
YOLO_MODELS = {
    'YOLOv8':  os.path.join(BASE, 'YOLOv8/runs/detect/yolov8_aircraft_test/weights/best.pt'),
    'YOLOv10': os.path.join(BASE, 'YOLOv10/runs/detect/yolov10_aircraft_finetune/weights/best.pt'),
    'YOLOv11': os.path.join(BASE, 'YOLOv11/runs/detect/yolo11_aircraft_finetune/weights/best.pt'),
    'YOLOv12': os.path.join(BASE, 'YOLOv12/runs/detect/runs/detect/yolo12l_aircraft_defects/weights/best.pt'),
    'RT-DETR': os.path.join(BASE, 'RT-DETR/runs/detect/rtdetr_aircraft_finetune/weights/best.pt'),
}

RCNN_WEIGHTS = {
    'Faster R-CNN': os.path.join(BASE, 'Faster RCNN/faster_rcnn_aircraft_epoch_10.pth'),
    'Mask R-CNN':   os.path.join(BASE, 'Masked RCNN/Masked RCNN/runs/maskrcnn/best_model.pth'),
}

CLASS_NAMES_JSON = os.path.join(
    BASE, 'Masked RCNN/Masked RCNN/runs/maskrcnn/class_names.json'
)

# ── Class names (YOLO order: 0-indexed) ──────────────────────────────────────
CLASSES = ['crack', 'dent', 'missing-head', 'paint-off', 'scratch']
NUM_CLASSES = len(CLASSES)  # 5

# ── Colour palette (one per model) ───────────────────────────────────────────
MODEL_COLORS = {
    'YOLOv8':       '#E63946',
    'YOLOv10':      '#F4A261',
    'YOLOv11':      '#228B48',
    'YOLOv12':      '#882C88',
    'RT-DETR':      '#3644AC',
    'Faster R-CNN': '#F4D03F',
    'Mask R-CNN':   '#00BCD4',
}

# Sanity-check key paths
for label, path in [('Full YAML', FULL_DATA_YAML), ('COCO JSON', COCO_VALID_JSON)]:
    print(f'{label}: {"✅ found" if os.path.exists(path) else "❌ MISSING — update path above"}')

Full YAML: ✅ found
COCO JSON: ✅ found


In [7]:
import yaml
import shutil

# ── Fix full_data.yaml: rewrite local paths → Colab Drive paths ───────────────

FIXED_YAML_PATH = '/content/full_data_colab.yaml'   # written to local Colab disk (fast)

with open(FULL_DATA_YAML, 'r') as f:
    yaml_content = yaml.safe_load(f)

print('Original YAML content:')
print(yaml_content)

# Replace the 'path' key with the correct Colab path
yaml_content['path'] = os.path.join(BASE, '_full_dataset_temp')

# 'train', 'val', 'test' are usually relative subpaths like 'images/train'
# If they are absolute local paths, strip and make them relative
for split in ['train', 'val', 'test']:
    if split in yaml_content and yaml_content[split]:
        val = yaml_content[split]
        if isinstance(val, str) and val.startswith('/'):
            # Extract just the relative part after '_full_dataset_temp/'
            if '_full_dataset_temp' in val:
                rel = val.split('_full_dataset_temp/')[-1]
            else:
                rel = os.path.basename(val)
            yaml_content[split] = rel
            print(f'Fixed {split}: {val!r} → {rel!r}')

with open(FIXED_YAML_PATH, 'w') as f:
    yaml.dump(yaml_content, f, default_flow_style=False)

print('\nFixed YAML written to:', FIXED_YAML_PATH)
print('Fixed YAML content:')
with open(FIXED_YAML_PATH) as f:
    print(f.read())

# Update the variable used by the eval cell
FULL_DATA_YAML = FIXED_YAML_PATH

Original YAML content:
{'names': ['crack', 'dent', 'missing-head', 'paint-off', 'scratch'], 'nc': 5, 'path': '/home/akash/Project Exhibition/_full_dataset_temp', 'val': 'images', 'train': 'images', 'test': 'images'}

Fixed YAML written to: /content/full_data_colab.yaml
Fixed YAML content:
names:
- crack
- dent
- missing-head
- paint-off
- scratch
nc: 5
path: /content/drive/MyDrive/Project Exhibition/_full_dataset_temp
test: images
train: images
val: images



## 3 · Global helpers & display style

In [8]:
import gc
import json
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display
from sklearn.metrics import roc_curve, auc, average_precision_score
from sklearn.preprocessing import label_binarize

plt.style.use('dark_background')
plt.rcParams.update({'grid.alpha': 0.4, 'axes.grid': True, 'figure.dpi': 120})


# ── Cell colour function (contrast-safe) ──────────────────────────────────────
def color_value(val):
    if val >= 0.97:   bg, fg = '#1a7a1a', '#ffffff'
    elif val >= 0.95: bg, fg = '#4caf50', '#ffffff'
    elif val >= 0.92: bg, fg = '#a5d6a7', '#1b1b1b'
    elif val >= 0.90: bg, fg = '#fff9c4', '#1b1b1b'
    elif val >= 0.87: bg, fg = '#ffb74d', '#1b1b1b'
    else:             bg, fg = '#e53935', '#ffffff'
    return f'background-color: {bg}; color: {fg}; font-weight: bold;'


# ── Table display ─────────────────────────────────────────────────────────────
TABLE_STYLES = [
    {'selector': 'caption',
     'props': [('font-size', '16px'), ('font-weight', 'bold'),
               ('text-align', 'left'), ('padding', '8px 0 6px 0'),
               ('color', '#f0f0f0')]},
    {'selector': 'th',
     'props': [('background-color', '#1c2833'), ('color', '#f0f0f0'),
               ('text-align', 'center'), ('padding', '8px 14px'),
               ('font-size', '13px'), ('letter-spacing', '0.5px')]},
    {'selector': 'th.row_heading',
     'props': [('background-color', '#2e4057'), ('color', '#ffffff'),
               ('text-align', 'left'), ('padding', '6px 16px'),
               ('font-style', 'italic')]},
    {'selector': 'td',
     'props': [('text-align', 'center'), ('padding', '6px 14px'),
               ('font-size', '13px'), ('border', '1px solid #cccccc')]},
    {'selector': 'tr:first-child th.row_heading',
     'props': [('background-color', '#145a32'), ('color', '#ffffff'),
               ('font-weight', 'bold'), ('font-style', 'normal')]},
]

METRIC_COLS = ['Precision', 'Recall', 'F1-Score', 'mAP50', 'Accuracy']


def display_table(model_name, rows):
    """Display a styled per-class metrics table."""
    df = pd.DataFrame(rows).set_index('Class')
    styled = (
        df.style
        .format('{:.4f}')
        .map(color_value, subset=METRIC_COLS)
        .set_caption(f'📊 {model_name} — Per-Class Metrics (Full Dataset)')
        .set_table_styles(TABLE_STYLES)
    )
    display(styled)


# ── GPU memory cleanup ─────────────────────────────────────────────────────────
def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


# ── Global store for aggregate metrics ────────────────────────────────────────
summary_rows = []   # filled by each section below
roc_data     = {}   # model_name -> (y_true_bin, y_score)

print('Helpers loaded ✅')

Helpers loaded ✅


## 4 · YOLO-family + RT-DETR — Per-class tables

> Each model is loaded, validated on the **full dataset**, then unloaded before the next one to avoid VRAM exhaustion.

In [ ]:
from ultralytics import YOLO, RTDETR

for model_name, weights_path in YOLO_MODELS.items():
    print('=' * 70)
    print(f'  Model: {model_name}')
    print('=' * 70)

    if not os.path.exists(weights_path):
        print(f'  ⚠️  Weights not found: {weights_path}\n')
        continue

    free_gpu()
    model = None
    try:
        model = (RTDETR if 'RT-DETR' in model_name else YOLO)(weights_path)

        # save_json=True → writes predictions.json needed later for ROC
        metrics = model.val(
            data=FULL_DATA_YAML,
            device=0,
            batch=8,
            save_json=True,
            plots=False,
            verbose=False,
        )

        # ── Build per-class rows ───────────────────────────────────────────────
        cls_names      = list(metrics.names.values())
        ap50_per_class = metrics.box.ap50.tolist()
        ap_per_class   = metrics.box.ap.tolist()
        p_per_class    = metrics.box.p.tolist()
        r_per_class    = metrics.box.r.tolist()

        rows = []
        for i, cls in enumerate(cls_names):
            p, r = p_per_class[i], r_per_class[i]
            f1   = 2*p*r/(p+r) if (p+r) > 0 else 0.0
            rows.append({
                'Class':     cls,
                'Precision': round(p, 4),
                'Recall':    round(r, 4),
                'F1-Score':  round(f1, 4),
                'mAP50':     round(ap50_per_class[i], 4),
                'Accuracy':  round(ap_per_class[i], 4),
            })

        mp, mr = metrics.box.mp, metrics.box.mr
        mf1    = 2*mp*mr/(mp+mr) if (mp+mr) > 0 else 0.0
        rows.insert(0, {
            'Class':     'all',
            'Precision': round(mp, 4),
            'Recall':    round(mr, 4),
            'F1-Score':  round(mf1, 4),
            'mAP50':     round(metrics.box.map50, 4),
            'Accuracy':  round(metrics.box.map, 4),
        })

        display_table(model_name, rows)

        # Save aggregate summary row
        summary_rows.append({
            'Model':     model_name,
            'Precision': round(mp, 4),
            'Recall':    round(mr, 4),
            'F1-Score':  round(mf1, 4),
            'mAP50':     round(metrics.box.map50, 4),
            'Accuracy':  round(metrics.box.map, 4),
        })

        # Save path to predictions.json for ROC cell
        pred_json = os.path.join(str(metrics.save_dir), 'predictions.json')
        if os.path.exists(pred_json):
            roc_data[model_name] = pred_json
            print(f'  ✅ predictions.json → {pred_json}')

    except Exception as e:
        print(f'  ❌ Error: {e}')
    finally:
        del model
        free_gpu()

    print()

  Model: YOLOv8
Ultralytics 8.4.14 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 11,127,519 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 32.5±71.1 ms, read: 0.0±0.0 MB/s, size: 27.0 KB)
val: Scanning /content/drive/MyDrive/Project Exhibition/_full_dataset_temp/labels... 4557 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4558/4558 1.9it/s 40:41
val: New cache created: /content/drive/MyDrive/Project Exhibition/_full_dataset_temp/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 570/570 8.9it/s 1:04
                   all       4558       4557      0.976      0.954      0.992      0.992
Speed: 1.0ms preprocess, 8.2ms inference, 0.0ms loss, 1.3ms postprocess per image
Saving /content/runs/detect/val6/predictions.json...
Results saved to /content/runs/detect/val6


,Precision,Recall,F1-Score,mAP50,Accuracy
Class,,,,,
all,0.9761,0.9543,0.9651,0.9924,0.9923
crack,0.9941,0.9583,0.9759,0.9931,0.9930
dent,0.9959,0.9102,0.9511,0.9932,0.9931
missing-head,0.9881,0.9699,0.9789,0.9936,0.9936
paint-off,0.9788,0.9591,0.9689,0.9903,0.9903
scratch,0.9236,0.9740,0.9481,0.9916,0.9916


  ✅ predictions.json → /content/runs/detect/val6/predictions.json

  Model: YOLOv10
Ultralytics 8.4.14 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv10n summary (fused): 102 layers, 2,266,143 parameters, 0 gradients, 6.5 GFLOPs
val: Fast image access ✅ (ping: 0.5±0.2 ms, read: 3.6±3.1 MB/s, size: 10.7 KB)
val: Scanning /content/drive/MyDrive/Project Exhibition/_full_dataset_temp/labels.cache... 4557 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4558/4558 1.4Git/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 570/570 4.7it/s 2:02
                   all       4558       4557      0.964      0.972      0.991       0.99
Speed: 1.1ms preprocess, 4.1ms inference, 0.0ms loss, 0.3ms postprocess per image
Saving /content/runs/detect/val7/predictions.json...
Results saved to /content/runs/detect/val7


,Precision,Recall,F1-Score,mAP50,Accuracy
Class,,,,,
all,0.9645,0.9724,0.9684,0.9911,0.9897
crack,0.9619,0.9743,0.9681,0.9907,0.9900
dent,0.9816,0.9810,0.9813,0.9941,0.9941
missing-head,0.9747,0.9731,0.9739,0.9936,0.9897
paint-off,0.9626,0.9561,0.9593,0.9889,0.9869
scratch,0.9416,0.9773,0.9591,0.9881,0.9880


  ✅ predictions.json → /content/runs/detect/val7/predictions.json

  Model: YOLOv11
Ultralytics 8.4.14 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 101 layers, 9,414,735 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 0.4±0.0 ms, read: 12.3±9.1 MB/s, size: 15.7 KB)
val: Scanning /content/drive/MyDrive/Project Exhibition/_full_dataset_temp/labels.cache... 4557 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4558/4558 910.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 570/570 4.3it/s 2:14
                   all       4558       4557       0.99      0.969      0.994      0.993
Speed: 1.0ms preprocess, 7.6ms inference, 0.0ms loss, 1.4ms postprocess per image
Saving /content/runs/detect/val8/predictions.json...
Results saved to /content/runs/detect/val8


,Precision,Recall,F1-Score,mAP50,Accuracy
Class,,,,,
all,0.9899,0.9688,0.9792,0.9936,0.9934
crack,0.9941,0.9524,0.9728,0.9927,0.9925
dent,0.9915,0.9848,0.9881,0.9946,0.9944
missing-head,0.9902,0.9818,0.9860,0.9948,0.9947
paint-off,0.9835,0.9469,0.9649,0.9922,0.9920
scratch,0.9900,0.9784,0.9842,0.9936,0.9935


  ✅ predictions.json → /content/runs/detect/val8/predictions.json

  Model: YOLOv12
Ultralytics 8.4.14 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv12l summary (fused): 283 layers, 26,342,927 parameters, 0 gradients, 88.6 GFLOPs
val: Fast image access ✅ (ping: 0.7±0.1 ms, read: 3.7±2.8 MB/s, size: 7.5 KB)
val: Scanning /content/drive/MyDrive/Project Exhibition/_full_dataset_temp/labels.cache... 4557 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4558/4558 955.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 570/570 2.1it/s 4:34
                   all       4558       4557       0.97      0.952      0.989      0.988
Speed: 1.0ms preprocess, 40.1ms inference, 0.0ms loss, 0.9ms postprocess per image
Saving /content/runs/detect/val9/predictions.json...
Results saved to /content/runs/detect/val9


,Precision,Recall,F1-Score,mAP50,Accuracy
Class,,,,,
all,0.9697,0.9524,0.9610,0.9891,0.9875
crack,0.9880,0.9328,0.9596,0.9878,0.9871
dent,0.9636,0.9551,0.9593,0.9886,0.9881
missing-head,0.9379,0.9942,0.9652,0.9942,0.9939
paint-off,0.9768,0.9258,0.9507,0.9883,0.9873
scratch,0.9822,0.9541,0.9679,0.9867,0.9812


  ✅ predictions.json → /content/runs/detect/val9/predictions.json

  Model: RT-DETR
Ultralytics 8.4.14 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary: 310 layers, 31,994,015 parameters, 0 gradients, 103.5 GFLOPs
val: Fast image access ✅ (ping: 0.5±0.1 ms, read: 10.2±7.2 MB/s, size: 13.0 KB)
val: Scanning /content/drive/MyDrive/Project Exhibition/_full_dataset_temp/labels.cache... 4557 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4558/4558 1.3Git/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 28% ━━━───────── 160/570 2.9it/s 2:08<2:20

## 5 · Faster R-CNN & Mask R-CNN — Per-class tables

Both models share the same COCO-format validation set. We run inference image-by-image and compute Precision, Recall, F1, and mAP50 using a lightweight IoU-matching loop (no extra dependencies).

In [10]:
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn, maskrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn  import MaskRCNNPredictor
from torchvision.transforms import functional as TF
from PIL import Image
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import shutil

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Load COCO ground truth ────────────────────────────────────────────────────
coco_gt = COCO(COCO_VALID_JSON)
cat_ids       = sorted(coco_gt.getCatIds())
coco_classes  = [coco_gt.loadCats(cid)[0]['name'] for cid in cat_ids]
print('COCO categories:', coco_classes)

# ── Copy validation images from Drive → local disk (fast I/O) ────────────────
LOCAL_IMG_DIR = '/content/rcnn_valid_images'
if not os.path.exists(LOCAL_IMG_DIR):
    print('Copying validation images to local disk (one-time, ~1-2 min)...')
    shutil.copytree(COCO_VALID_IMAGES, LOCAL_IMG_DIR)
    print(f'✅ Copied to {LOCAL_IMG_DIR}')
else:
    print(f'✅ Local image cache already exists: {LOCAL_IMG_DIR}')

img_count = len([f for f in os.listdir(LOCAL_IMG_DIR)
                 if f.lower().endswith(('.jpg','.jpeg','.png'))])
print(f'   Images available: {img_count}')


# ── Auto-detect num_classes from checkpoint ───────────────────────────────────
def detect_num_classes(weights_path):
    """
    Read cls_score.bias from checkpoint to determine how many classes
    (including background) the model was trained with.
    Returns num_classes_with_background.
    """
    ckpt  = torch.load(weights_path, map_location='cpu')
    state = ckpt.get('model', ckpt) if isinstance(ckpt, dict) else ckpt
    # Try standard key first, then common variants
    for key in ['roi_heads.box_predictor.cls_score.bias',
                'module.roi_heads.box_predictor.cls_score.bias']:
        if key in state:
            return state[key].shape[0]   # includes background
    raise KeyError('Cannot find cls_score.bias in checkpoint — check key names.')


# ── Build model with correct head size ───────────────────────────────────────
def build_model(arch, weights_path):
    num_cls_with_bg = detect_num_classes(weights_path)
    print(f'  Detected classes (incl. background): {num_cls_with_bg}')

    ckpt  = torch.load(weights_path, map_location='cpu')
    state = ckpt.get('model', ckpt) if isinstance(ckpt, dict) else ckpt

    if arch == 'faster':
        m = fasterrcnn_resnet50_fpn(weights=None)
        in_feat = m.roi_heads.box_predictor.cls_score.in_features
        m.roi_heads.box_predictor = FastRCNNPredictor(in_feat, num_cls_with_bg)
    else:
        m = maskrcnn_resnet50_fpn(weights=None)
        in_feat = m.roi_heads.box_predictor.cls_score.in_features
        m.roi_heads.box_predictor = FastRCNNPredictor(in_feat, num_cls_with_bg)
        in_feat_mask = m.roi_heads.mask_predictor.conv5_mask.in_channels
        m.roi_heads.mask_predictor = MaskRCNNPredictor(in_feat_mask, 256, num_cls_with_bg)

    m.load_state_dict(state, strict=True)
    m.eval().to(DEVICE)
    return m, num_cls_with_bg


# ── Batch inference with local images ────────────────────────────────────────
@torch.no_grad()
def run_coco_eval(model, coco_gt, local_img_dir, cat_ids, score_thresh=0.05):
    img_ids   = coco_gt.getImgIds()
    all_preds = []
    missing   = 0

    for img_id in img_ids:
        info     = coco_gt.loadImgs(img_id)[0]
        img_path = os.path.join(local_img_dir, info['file_name'])

        if not os.path.exists(img_path):
            missing += 1
            continue

        img    = Image.open(img_path).convert('RGB')
        tensor = TF.to_tensor(img).unsqueeze(0).to(DEVICE)
        out    = model(tensor)[0]

        for box, lbl, score in zip(out['boxes'], out['labels'], out['scores']):
            if score < score_thresh:
                continue
            lbl_int = int(lbl)
            # label 0 = background; labels are 1-indexed into cat_ids
            if lbl_int == 0 or lbl_int > len(cat_ids):
                continue
            cat_id = cat_ids[lbl_int - 1]
            x1, y1, x2, y2 = box.tolist()
            all_preds.append({
                'image_id':    img_id,
                'category_id': cat_id,
                'bbox':        [x1, y1, x2 - x1, y2 - y1],
                'score':       float(score),
            })

    print(f'  Images processed: {len(img_ids) - missing}/{len(img_ids)}  |  '
          f'Predictions: {len(all_preds)}')

    if not all_preds:
        print('  ⚠️  No predictions — check score_thresh or model head.')
        return None, None

    coco_dt   = coco_gt.loadRes(all_preds)
    evaluator = COCOeval(coco_gt, coco_dt, iouType='bbox')
    evaluator.evaluate()
    evaluator.accumulate()
    evaluator.summarize()
    return evaluator, all_preds


# ── Extract per-class metrics from COCOeval ───────────────────────────────────
def coco_eval_to_rows(evaluator, coco_classes):
    precision = evaluator.eval['precision']   # T x R x K x A x M
    recalls   = evaluator.eval['recall']      # T x K x A x M

    rows, ap50_all = [], []
    for k, cls_name in enumerate(coco_classes):
        pr   = precision[0, :, k, 0, 2]
        pr   = pr[pr > -1]
        ap50 = float(np.mean(pr)) if len(pr) else 0.0

        r    = float(recalls[0, k, 0, 2]) if recalls[0, k, 0, 2] > -1 else 0.0
        p    = min(ap50 / r, 1.0) if r > 0 else 0.0
        f1   = 2*p*r/(p+r) if (p+r) > 0 else 0.0

        ap50_all.append(ap50)
        rows.append({
            'Class':     cls_name,
            'Precision': round(p, 4),
            'Recall':    round(r, 4),
            'F1-Score':  round(f1, 4),
            'mAP50':     round(ap50, 4),
            'Accuracy':  round(ap50, 4),
        })

    mean_ap50 = float(np.mean(ap50_all))
    mr = float(np.mean([r['Recall']   for r in rows]))
    mf = float(np.mean([r['F1-Score'] for r in rows]))
    mp = float(np.mean([r['Precision'] for r in rows]))
    rows.insert(0, {
        'Class':     'all',
        'Precision': round(mp, 4),
        'Recall':    round(mr, 4),
        'F1-Score':  round(mf, 4),
        'mAP50':     round(mean_ap50, 4),
        'Accuracy':  round(mean_ap50, 4),
    })
    return rows


# ── Evaluate both R-CNN variants ──────────────────────────────────────────────
RCNN_ARCH = {
    'Faster R-CNN': 'faster',
    'Mask R-CNN':   'mask',
}

for model_name, arch in RCNN_ARCH.items():
    weights_path = RCNN_WEIGHTS[model_name]
    print('=' * 70)
    print(f'  Model: {model_name}')
    print('=' * 70)

    if not os.path.exists(weights_path):
        print(f'  ⚠️  Weights not found: {weights_path}\n')
        continue

    free_gpu()
    model = None
    try:
        model, num_cls_bg = build_model(arch, weights_path)

        # Use only the cat_ids that match the trained head (exclude background)
        effective_cat_ids = cat_ids[:num_cls_bg - 1]
        effective_classes = coco_classes[:num_cls_bg - 1]

        evaluator, all_preds = run_coco_eval(
            model, coco_gt, LOCAL_IMG_DIR, effective_cat_ids)

        if evaluator is None:
            continue

        rows = coco_eval_to_rows(evaluator, effective_classes)
        display_table(model_name, rows)

        all_row = rows[0]
        summary_rows.append({
            'Model':     model_name,
            'Precision': all_row['Precision'],
            'Recall':    all_row['Recall'],
            'F1-Score':  all_row['F1-Score'],
            'mAP50':     all_row['mAP50'],
            'Accuracy':  all_row['Accuracy'],
        })

        roc_data[model_name] = ('rcnn', all_preds, coco_gt, effective_cat_ids)
        print(f'  ✅ {model_name} complete.\n')

    except Exception as e:
        import traceback
        print(f'  ❌ Error evaluating {model_name}:')
        traceback.print_exc()

    finally:
        del model
        free_gpu()

loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
COCO categories: ['different-surface-defects', 'crack', 'dent', 'missing-head', 'paint-off', 'scratch']
Copying validation images to local disk (one-time, ~1-2 min)...
✅ Copied to /content/rcnn_valid_images
   Images available: 148
  Model: Faster R-CNN
  Detected classes (incl. background): 6
  Images processed: 148/148  |  Predictions: 306
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.11s).
Accumulating evaluation results...
DONE (t=0.05s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.015
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.022
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.014
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.9

,Precision,Recall,F1-Score,mAP50,Accuracy
Class,,,,,
all,0.1180,0.1292,0.0976,0.0221,0.0221
different-surface-defects,0.0000,0.0000,0.0000,0.0000,0.0000
crack,0.1774,0.4419,0.2532,0.0784,0.0784
dent,0.0683,0.0870,0.0765,0.0059,0.0059
missing-head,0.0261,0.0370,0.0306,0.0010,0.0010
paint-off,0.3182,0.0800,0.1279,0.0255,0.0255


  ✅ Faster R-CNN complete.

  Model: Mask R-CNN
  ❌ Error evaluating Mask R-CNN:


Traceback (most recent call last):
  File "/tmp/ipykernel_1212/3066838244.py", line 184, in <cell line: 0>
    model, num_cls_bg = build_model(arch, weights_path)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_1212/3066838244.py", line 53, in build_model
    num_cls_with_bg = detect_num_classes(weights_path)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_1212/3066838244.py", line 48, in detect_num_classes
    raise KeyError('Cannot find cls_score.bias in checkpoint — check key names.')
KeyError: 'Cannot find cls_score.bias in checkpoint — check key names.'


## 6 · Aggregate comparison table & bar chart

In [ ]:
# ── Styled summary table ───────────────────────────────────────────────────────
df_summary = pd.DataFrame(summary_rows).set_index('Model')
print('\n=== Overall Model Comparison ===')

styled_summary = (
    df_summary.style
    .format('{:.4f}')
    .map(color_value, subset=METRIC_COLS)
    .set_caption('📊 All Models — Overall Metrics (Full Dataset)')
    .set_table_styles(TABLE_STYLES)
    .highlight_max(axis=0, color='#00e676', subset=METRIC_COLS)
)
display(styled_summary)

# Save CSV
df_summary.to_csv(os.path.join(BASE, 'model_comparison_colab.csv'))
print('Saved model_comparison_colab.csv')

In [ ]:
# ── Grouped bar chart ─────────────────────────────────────────────────────────
metrics_to_plot = ['Precision', 'Recall', 'F1-Score', 'mAP50', 'Accuracy']
models     = df_summary.index.tolist()
n_models   = len(models)
n_metrics  = len(metrics_to_plot)

x      = np.arange(n_metrics)
width  = 0.12
offsets = np.linspace(-(n_models-1)/2, (n_models-1)/2, n_models) * width

fig, ax = plt.subplots(figsize=(13, 6))
fig.patch.set_facecolor('#0d0d1a')
ax.set_facecolor('#0d0d1a')

for i, model_name in enumerate(models):
    vals = [df_summary.loc[model_name, m] for m in metrics_to_plot]
    bars = ax.bar(x + offsets[i], vals, width,
                  label=model_name,
                  color=MODEL_COLORS.get(model_name, '#aaaaaa'),
                  edgecolor='#222222', linewidth=0.5)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{v:.3f}', ha='center', va='bottom',
                fontsize=5.5, color='white', rotation=90)

ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot, fontsize=11, color='white')
ax.set_ylim(0.80, 1.03)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
ax.tick_params(colors='white')
ax.spines[['top','right']].set_visible(False)
ax.spines[['left','bottom']].set_color('#555555')
ax.set_ylabel('Score', fontsize=12, color='white')
ax.legend(loc='lower right', fontsize=9,
          framealpha=0.3, facecolor='#1a1a2e', labelcolor='white')
ax.set_title('Model Comparison — Aircraft Skin Defect Detection',
             color='white', fontsize=13, fontweight='bold', pad=14)
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'model_comparison_bar_colab.png'),
            dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

In [ ]:
# ── Radar / Spider chart ───────────────────────────────────────────────────────
from matplotlib.patches import FancyArrowPatch

categories = metrics_to_plot
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]  # close the loop

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
fig.patch.set_facecolor('#0d0d1a')
ax.set_facecolor('#0d0d1a')

for model_name in models:
    vals = [df_summary.loc[model_name, m] for m in categories]
    vals += vals[:1]
    color = MODEL_COLORS.get(model_name, '#aaaaaa')
    ax.plot(angles, vals, linewidth=2, linestyle='solid', label=model_name, color=color)
    ax.fill(angles, vals, alpha=0.08, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, color='white', fontsize=11)
ax.set_ylim(0.80, 1.0)
ax.yaxis.set_tick_params(labelcolor='grey', labelsize=8)
ax.spines['polar'].set_color('#444444')
ax.grid(color='#333333', linewidth=0.8)
ax.set_title('Radar Chart — Model Performance Overview',
             color='white', fontsize=13, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15),
          fontsize=9, framealpha=0.3, facecolor='#1a1a2e', labelcolor='white')
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'model_comparison_radar_colab.png'),
            dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

## 7 · Macro-average AUC-ROC curves

In [ ]:
# ── Helper: strip Roboflow hash from stem (YOLO predictions.json) ─────────────
def to_base_stem(s):
    s = os.path.splitext(s)[0]
    parts = s.split('.rf.')
    return parts[0] + '.rf'


# ── Ground truth from YOLO-format .txt labels ─────────────────────────────────
gt_by_base = {}
for fpath in glob.glob(os.path.join(VALID_LABEL_DIR, '*.txt')):
    stem = os.path.splitext(os.path.basename(fpath))[0]
    base = to_base_stem(stem)
    with open(fpath) as lf:
        lines = [l.strip() for l in lf if l.strip()]
    if lines:
        gt_by_base[base] = int(lines[0].split()[0])

print(f'GT samples loaded: {len(gt_by_base)}')


def load_yolo_predictions_json(json_path):
    """Load a YOLO-generated COCO predictions.json and match to GT."""
    with open(json_path) as f:
        preds = json.load(f)

    all_cats = {p['category_id'] for p in preds}
    offset   = 1 if min(all_cats) == 1 else 0

    img_class_score = {}
    for p in preds:
        base = to_base_stem(p['image_id'])
        cid  = p['category_id'] - offset
        sc   = p['score']
        key  = (base, cid)
        img_class_score[key] = max(img_class_score.get(key, 0.0), sc)

    y_true, y_score = [], []
    for base, true_cat in gt_by_base.items():
        scores_row = [img_class_score.get((base, c), 0.0) for c in range(NUM_CLASSES)]
        y_score.append(scores_row)
        y_true.append(true_cat)

    matched = sum(1 for base in gt_by_base
                  if any((base, c) in img_class_score for c in range(NUM_CLASSES)))
    print(f'  Matched: {matched}/{len(gt_by_base)}')

    y_true_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
    return y_true_bin, np.array(y_score)


def load_rcnn_predictions(all_preds_list, coco_gt_obj, cat_ids_list):
    """
    Build y_true_bin / y_score matrices from raw COCO-format R-CNN predictions
    matched against coco_gt ground truth.
    """
    img_ids    = coco_gt_obj.getImgIds()
    cat_id_to_idx_local = {cid: i for i, cid in enumerate(cat_ids_list)}
    nc = len(cat_ids_list)

    # Build GT per image
    gt_img = {}  # img_id -> set of class indices
    for img_id in img_ids:
        ann_ids = coco_gt_obj.getAnnIds(imgIds=img_id)
        anns    = coco_gt_obj.loadAnns(ann_ids)
        gt_img[img_id] = set(cat_id_to_idx_local[a['category_id']]
                              for a in anns if a['category_id'] in cat_id_to_idx_local)

    # Build prediction scores per image per class
    pred_scores = {img_id: [0.0]*nc for img_id in img_ids}
    for p in all_preds_list:
        if p['category_id'] not in cat_id_to_idx_local:
            continue
        cidx = cat_id_to_idx_local[p['category_id']]
        pred_scores[p['image_id']][cidx] = max(
            pred_scores[p['image_id']][cidx], p['score'])

    y_true, y_score = [], []
    for img_id in img_ids:
        for cls_idx in gt_img.get(img_id, []):
            row = [0] * nc
            row[cls_idx] = 1
            y_true.append(row)
            y_score.append(pred_scores[img_id])

    return np.array(y_true), np.array(y_score)


def macro_roc(y_true_bin, y_score, nc):
    fpr_all, tpr_all = [], []
    for i in range(nc):
        if y_true_bin[:, i].sum() == 0:
            continue
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_score[:, i])
        fpr_all.append(fpr)
        tpr_all.append(tpr)
    mean_fpr = np.linspace(0, 1, 300)
    mean_tpr = np.mean([np.interp(mean_fpr, f, t) for f, t in zip(fpr_all, tpr_all)], axis=0)
    mean_tpr[0] = 0.0
    return mean_fpr, mean_tpr, auc(mean_fpr, mean_tpr)


# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 7))
fig.patch.set_facecolor('#0d0d1a')
ax.set_facecolor('#0d0d1a')

for model_name, data in roc_data.items():
    try:
        if isinstance(data, str):  # YOLO predictions.json path
            y_true_bin, y_score = load_yolo_predictions_json(data)
            nc = NUM_CLASSES
        else:                      # R-CNN tuple
            _, all_preds_list, coco_gt_obj, cat_ids_list = data
            y_true_bin, y_score = load_rcnn_predictions(
                all_preds_list, coco_gt_obj, cat_ids_list)
            nc = len(cat_ids_list)

        fpr, tpr, roc_auc = macro_roc(y_true_bin, y_score, nc)
        color = MODEL_COLORS.get(model_name, '#aaaaaa')
        ax.plot(fpr, tpr, lw=2, color=color,
                label=f'{model_name}  (AUC = {roc_auc:.3f})')

    except Exception as e:
        print(f'[{model_name}] ROC skipped — {e}')

ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random classifier')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.02])
ax.set_xlabel('False Positive Rate', fontsize=12, color='white')
ax.set_ylabel('True Positive Rate', fontsize=12, color='white')
ax.tick_params(colors='white')
ax.spines[['top','right']].set_visible(False)
ax.spines[['left','bottom']].set_color('#555555')
ax.set_title('Macro-Average AUC-ROC — All Models\nAircraft Skin Defect Detection',
             fontsize=13, fontweight='bold', color='white')
ax.legend(loc='lower right', fontsize=10,
          framealpha=0.3, facecolor='#1a1a2e', labelcolor='white')
ax.grid(alpha=0.3, color='#444444')
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'AUC_ROC_comparison_colab.png'),
            dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

## 8 · Per-class mAP50 heatmap

In [ ]:
# ── Re-run fast validation (no save_json) to collect per-class AP50 for heatmap
# This cell uses the already-loaded metrics saved in per_class_ap50 dict.
# If you skipped cell 4, re-run it first.
#
# We build the heatmap from summary_rows + per-class data.
# For simplicity, re-evaluate each YOLO model quickly (plots=False, no json).

heatmap_data = {}  # model_name -> {class_name: ap50}

for model_name, weights_path in YOLO_MODELS.items():
    if not os.path.exists(weights_path):
        continue
    free_gpu()
    model = None
    try:
        model = (RTDETR if 'RT-DETR' in model_name else YOLO)(weights_path)
        metrics = model.val(
            data=FULL_DATA_YAML, device=0, batch=8,
            save_json=False, plots=False, verbose=False)
        cls_names = list(metrics.names.values())
        heatmap_data[model_name] = {
            cls: round(float(ap50), 4)
            for cls, ap50 in zip(cls_names, metrics.box.ap50.tolist())
        }
        print(f'  ✅ {model_name} heatmap data collected')
    except Exception as e:
        print(f'  ❌ {model_name}: {e}')
    finally:
        del model
        free_gpu()

# Add R-CNN rows if coco_eval was run
for model_name in ['Faster R-CNN', 'Mask R-CNN']:
    row = next((r for r in summary_rows if r['Model'] == model_name), None)
    if row:
        # We only have overall mAP50; fill all classes with that value as placeholder
        heatmap_data[model_name] = {c: row['mAP50'] for c in coco_classes}

if heatmap_data:
    df_heat = pd.DataFrame(heatmap_data).T  # models × classes
    df_heat = df_heat.reindex(columns=sorted(df_heat.columns))

    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor('#0d0d1a')
    sns.heatmap(
        df_heat, annot=True, fmt='.3f', cmap='RdYlGn',
        vmin=0.85, vmax=1.0, linewidths=0.5, linecolor='#333333',
        ax=ax, annot_kws={'size': 10},
        cbar_kws={'shrink': 0.8}
    )
    ax.set_title('Per-Class mAP50 Heatmap — All Models',
                 color='white', fontsize=13, fontweight='bold', pad=12)
    ax.set_xlabel('Defect Class', fontsize=11, color='white')
    ax.set_ylabel('Model', fontsize=11, color='white')
    ax.tick_params(colors='white', labelsize=10)
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig(os.path.join(BASE, 'per_class_map50_heatmap_colab.png'),
                dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()
else:
    print('No heatmap data — ensure at least one model was evaluated.')

---
## ✅ All outputs saved to Google Drive
| File | Content |
|---|---|
| `model_comparison_colab.csv` | Aggregate metrics table |
| `model_comparison_bar_colab.png` | Grouped bar chart |
| `model_comparison_radar_colab.png` | Radar / spider chart |
| `AUC_ROC_comparison_colab.png` | Macro-average ROC curves |
| `per_class_map50_heatmap_colab.png` | Per-class mAP50 heatmap |